<a href="https://colab.research.google.com/github/Malaikahaamer/Cheating_detection/blob/main/AI_Based_Live_Cheating_Detection_in_Exam_Hall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install ultralytics roboflow timm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.9 MB/s eta 0:00:00


In [22]:
import torch
import torch.nn as nn
from ultralytics import YOLO

In [23]:
from roboflow import Roboflow

rf = Roboflow(api_key="HXSRr1fACVXMcg38dO1e")
project = rf.workspace("cheating-detection-bp8bo").project("exam-cheating-detector-v3-kym7j")
version = project.version(7)
dataset = version.download("yolov8")

print("Dataset downloaded at:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Exam-Cheating-Detector-V3-7 in yolov8:: 100%|██████████| 2631/2631 [00:00<00:00, 5443.91it/s]

Dataset downloaded at: /content/Exam-Cheating-Detector-V3-7


In [24]:
class ECA(nn.Module):
    def __init__(self, channels, k_size=3):
        super(ECA, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size,
                              padding=(k_size-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)

In [25]:
model = YOLO("yolov8n.pt")

In [26]:
import torch.nn as nn

def add_eca(model):
    for i, layer in enumerate(model.model.model):
        if hasattr(layer, "cv2"):
            try:
                ch = layer.cv2.out_channels
                layer.eca = ECA(ch)
            except:
                pass

add_eca(model)
print("ECA injected into model")

ECA injected into model


In [27]:
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0  # use GPU in Colab
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Exam-Cheating-Detector-V3-7/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

In [28]:
model.export(format="onnx")

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 13, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 339ms
Prepared 4 packages in 4.15s
Installed 4 packages in 340ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.91

requirements: AutoUpdate success ✅ 5.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimm

'/content/runs/detect/train/weights/best.onnx'

In [29]:
model.save("yolov8_eca_cheating.pt")

In [40]:
results = model.predict(
    source="7092118-hd_1080_1920_30fps.mp4",  # or video.mp4 / webcam
    save=True,
    conf=0.5
)


image 1/1 /content/gettyimages-80411047-612x612.jpg: 448x640 (no detections), 45.4ms
Speed: 2.7ms preprocess, 45.4ms inference, 0.7ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict


In [ ]:
from google.colab import files
uploaded = files.upload()